# M6 · 组合优化与独立画质门禁

**核心问题：** 最快就代表可发布吗？重算 B0/BEST 6.12×，区分 LPIPS 感知距离与保真度，并保留 BEST 的独立 `PENDING_HUMAN_REVIEW` 门禁。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M06_combined_quality/LESSON.zh-CN.md](../course/M06_combined_quality/LESSON.zh-CN.md)。切换语言：[English](./07_combined_and_quality.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
rows = ns.summary_rows()
b, best = rows["B0"], rows["BEST"]
ratio = b["warm_total_median_s"] / best["warm_total_median_s"]
print(f"Measured warm B0={b['warm_total_median_s']:.3f}s; BEST={best['warm_total_median_s']:.3f}s; gain={ratio:.3f}x")
assert 6.0 < ratio < 6.3
print("BEST changes model variant, schedule AND precision. This is not a lossless kernel swap.")
plot_values = {"B0": b["warm_total_median_s"], "BEST": best["warm_total_median_s"]}
plot_ylabel, plot_title = "warm seconds", "Combined result requires separate image review"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
quality = ns.read_json("evidence/quality/BEST/quality.json")
assert quality["semantics"] == "distance"
print("LPIPS is a DISTANCE here, not an objective quality grade.")
for prompt_id, value in quality["median"].items():
    print(f"{prompt_id}: {value:.3f} (distance)")
claims = ns.read_json("course/data/verified_claims.json")["claims"]
pending = [x for x in claims if "BEST" in x["id"] and "PENDING" in str(x.get("human_quality_verdict", ""))]
assert pending, "An independent BEST human-quality review is required"
print("BEST quality gate is PENDING; do not mark PASS automatically.")


In [ ]:
# Human-readable side-by-side visual review is optional in a real Jupyter kernel.
SHOW_QUALITY_IMAGES = False
if SHOW_QUALITY_IMAGES:
    from IPython.display import display, Image
    display(Image(filename=str(ROOT / "evidence/quality/BEST/quality_grid.png")))
    display(Image(filename=str(ROOT / "evidence/quality/BEST/quality_crops_poster.png")))
else:
    print("Image display is OFF. Set SHOW_QUALITY_IMAGES=True to inspect grids/native poster crops.")
print("Looking at images does NOT automatically create an owner verdict.")


## 可选 GPU 真机冒烟——必须显式授权

需要 ROCm 环境、模型与 GPU 额度。CI 不执行；结果只存于 `runs/`，不得混入官方证据。

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("BEST", "notebook-best", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## 学生思考 / 自检

打开 `evidence/quality/BEST/quality_crops_poster.png`，按原生分辨率检查中文标题排版。制定三级人工审阅标准；Notebook 不会替 Owner 写入 PASS。

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
